# Generation of LCA indicators and associated .mod and .dat files

In [ ]:
# %pip install brightway2
# %pip install mescal
# %pip install energyscope

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import pandas as pd
import bw2data as bd
from mescal import *
from energyscope.models import Model
from energyscope.energyscope import Energyscope
from energyscope.result import postprocessing
from utils import *
from shared.utils import run_model, load_snapshot

In [ ]:
ei_version = '3.10.1'
year = 2023 # 2023 or 2050
ssp_rcp = 'SSP5-H' # 'SSP2-L' or 'SSP5-H'
reg_level = 'base' # can be 'base_wo_iam', 'base', 'spat', 'spat_fore', 'spat_fore_back'

| Assessment type | IAM assumptions | Spatialization | Foreground regionalization | Background regionalization |
|-----------------|-----------------|----------------|----------------------------|----------------------------|
| base_wo_iam     |                 |                |                            |                            |
| base            | x               |                |                            |                            |
| spat            | x               | x              |                            |                            |
| spat_fore       | x               | x              | x                          |                            |
| spat_fore_back  | x               | x              | x                          | x                          |

The assessment type `base_wo_iam` and the column 'IAM assumptions' only applies when `year` is set to 2050.

In [ ]:
path_inputs = '../01_Notebooks/Data/'
path_data = f'../02_AMPL_files/data/{year}/'
path_model = '../02_AMPL_files/model/'
path_data_lca = path_data + f'{reg_level}/'
if year == 2050:
    path_results = f'../03_Results/LCA/{year}/{reg_level}/{ssp_rcp}/'
else:
    path_results = f'../03_Results/LCA/{year}/{reg_level}/'

## Initialize the EnergyScope model

In [ ]:
# AMPL licence 
path_to_ampl_licence = r'C:\Users\matth\ampl' # Path to the AMPL license file
os.environ['PATH'] = path_to_ampl_licence+':'+os.environ['PATH']

In [ ]:
# Initialize the 2023 QC model with .mod and .dat files
model = load_snapshot(year)

In [ ]:
# Solve the model and get results
results = run_model(model)

## Load data and initialize the ESM class

In [ ]:
# Load the data
mapping = pd.read_csv(path_inputs+f'mapping.csv')
unit_conversion = pd.read_excel(path_inputs+'unit_conversion.xlsx')  # open file and press enter in one computation cell to avoid misreading
techno_compositions = pd.read_csv(path_inputs+'technology_compositions.csv')
tech_specifics = pd.read_csv(path_inputs+'technology_specifics.csv')
efficiency = pd.read_csv(path_inputs+'efficiency.csv')
lifetime = pd.read_csv(path_inputs+'lifetime.csv')
mapping_es_flows_to_cpc = pd.read_csv(path_inputs+'mapping_esm_flows_to_CPC.csv')
impact_abbrev = pd.read_csv(path_inputs+'impact_abbrev.csv')
mapping_new_product_to_cpc = pd.read_csv(path_inputs + 'mapping_new_products_to_CPC.csv')

In [ ]:
# Load the model from energyscope model
model = results.parameters['layers_in_out'].reset_index().rename(columns={'index0': 'Name', 'index1': 'Flow', 'layers_in_out': 'Amount'}).drop(columns=['Run'])
model = model[model['Amount'] != 0]
# model.to_csv(path_inputs+f'model_{year}.csv', index=False)

In [ ]:
# Set up your Brightway project
bd.projects.set_current(f'ecoinvent{ei_version}')

In [ ]:
# Database names
if reg_level == 'base':
    name_main_database = f'ecoinvent_cutoff_{ei_version}_image_{ssp_rcp}_{year}+truck_carculator'
elif reg_level == 'base_wo_iam':
    name_main_database = f'ecoinvent_cutoff_{ei_version}_image_{ssp_rcp}_{year}_wo_updates+truck_carculator'
elif reg_level in ['spat', 'spat_fore']:
    name_main_database = f'ei_{ei_version}_image_{ssp_rcp}_{year}+truck_carculator_reg_wo'
elif reg_level == 'spat_fore_back':
    name_main_database = f'ei_{ei_version}_image_{ssp_rcp}_{year}+truck_carculator_reg'
else:
    raise ValueError(f"Unknown assessment type: {reg_level}.")

name_regioinvent_db = f'regiopremise_{ei_version}_image_{ssp_rcp}_{year}+truck_carculator'
name_biosphere_db = 'biosphere3'

if reg_level in ['base', 'base_wo_iam']:
    name_spatialized_biosphere_db = None
    spatialized_biosphere_db = None
else:
    name_spatialized_biosphere_db = 'biosphere3_spatialized_flows'
    spatialized_biosphere_db = Database(name_spatialized_biosphere_db)

name_es_database = f'EnergyScope_CA-QC_{year}_{reg_level}_{ssp_rcp}'

In [ ]:
if reg_level in ['base', 'base_wo_iam', 'spat']:
    regionalize_foregrounds = None
elif reg_level in ['spat_fore', 'spat_fore_back']:
    regionalize_foregrounds = ['Operation', 'Resource']
else:
    raise ValueError(f"Unknown assessment type: {reg_level}.")

In [ ]:
if reg_level in ['spat_back', 'spat_fore_back']:
    main_db = Database(db_names=[name_main_database, name_regioinvent_db], create_pickle=True)
else:
    main_db = Database(db_names=name_main_database, create_pickle=True)

In [ ]:
ranking_best_ecoinvent_locations_for_QC = [
    'CA-QC', # Quebec
    'CAN', # Canada in IMAGE
    'CA', # Canada
    'CAZ', # Canada - Australia - New Zealand in REMIND
    'RNA', # North America
    'US', # United States
    'USA', # United States in REMIND and IMAGE
    'GLO', # Global average 
    'RoW', # Rest of the world
]

In [ ]:
# Add CPC categories to the main database
main_db.add_CPC_categories(mapping_new_products_to_CPC=mapping_new_product_to_cpc, overwrite_existing_CPC=True)

In [ ]:
# Change the main database name if needed
if mapping['Database'].iloc[0] != name_main_database:
    mapping['Database'] = len(mapping) * [name_main_database]

In [ ]:
esm = ESM(
    # Mandatory inputs
    mapping=mapping,
    unit_conversion=unit_conversion,
    model=model,
    mapping_esm_flows_to_CPC_cat=mapping_es_flows_to_cpc,
    main_database=main_db,
    esm_db_name=name_es_database,
    
    # Optional inputs
    technology_compositions=techno_compositions,
    tech_specifics=tech_specifics,
    lifetime=lifetime,
    efficiency=efficiency,
    regionalize_foregrounds=regionalize_foregrounds,
    accepted_locations=['CA-QC'],
    locations_ranking=ranking_best_ecoinvent_locations_for_QC,
    esm_location='CA-QC',
    results_path_file=path_results,
    biosphere_db_name=name_biosphere_db,
    
    # If we want regionalized results 
    spatialized_biosphere_db=spatialized_biosphere_db,
)

In [ ]:
esm.clean_inputs()

In [ ]:
esm.check_inputs()

In [ ]:
# Adapt mapping file to ESM location
esm.change_location_mapping_file()
# esm.mapping.to_csv(path_inputs+f'mapping.csv', index=False)

In [ ]:
missing_flows = main_db.test_mapping_file(esm.mapping)

In [ ]:
main_db = {}  # Free memory

### Generate ESM database

In [ ]:
# Foreground regionalization, double-counting removal, and efficiency harmonization
esm_db = esm.create_esm_database(write_database=False, return_database=True)

## Generate LCA metrics

In [ ]:
methods = [
    'IMPACT World+ Midpoint 2.1_regionalized for ecoinvent v3.10',  # also works with non-spatialized datasets
    'IMPACT World+ Damage 2.1_regionalized for ecoinvent v3.10',  # also works with non-spatialized datasets
    'IMPACT World+ Damage 2.1 for ecoinvent v3.10 (incl. CO2 uptake)',
    'IMPACT World+ Midpoint 2.1 for ecoinvent v3.10 (incl. CO2 uptake)',
]

### Life-cycle emissions

In [ ]:
contrib_analysis = 'both'  # 'emissions', 'processes', 'both' or None

In [ ]:
# LCIA, Lifetime harmonization
if contrib_analysis is not None:
    R_long, contrib_analysis_res, _ = esm.compute_impact_scores(
        methods=methods,
        impact_abbrev=impact_abbrev,
        contribution_analysis=contrib_analysis,
        contribution_analysis_limit_type='number',
        contribution_analysis_limit=30,
    )
    contrib_analysis_emissions = contrib_analysis_res[contrib_analysis_res.database.str.contains('biosphere')]
    contrib_analysis_processes = contrib_analysis_res[~contrib_analysis_res.database.str.contains('biosphere')]
    contrib_analysis_emissions.to_csv(f'{path_results}contribution_analysis_emissions.csv', index=False)
    contrib_analysis_processes.to_csv(f'{path_results}contribution_analysis_processes.csv', index=False)
else:
    R_long, contrib_analysis_res, _ = esm.compute_impact_scores(
        methods=methods,
        impact_abbrev=impact_abbrev,
    )

In [ ]:
# Set the impact of transformers to zero for operation
R_long['Value'] = R_long.apply(lambda row: 0 if row['Name'].startswith('TRAFO_') and row['Type'] == 'Operation' else row['Value'], axis=1)

In [ ]:
R_long.to_csv(f'{path_results}impact_scores.csv', index=False) # [impact / kW(h) or pkm(/h) or tkm(/h)]

### Direct emissions

In [ ]:
R_long_direct_emissions, contrib_analysis_direct_emissions, _ = esm.compute_impact_scores(
    methods=methods,
    assessment_type='direct emissions',
    impact_abbrev=impact_abbrev,
    overwrite=True,
    contribution_analysis='emissions',
    contribution_analysis_limit=10,
)

In [ ]:
contrib_analysis_direct_emissions.to_csv(f'{path_results}contribution_analysis_direct_emissions.csv', index=False)

In [ ]:
# Set the impact of transformers to zero for operation
R_long_direct_emissions['Value'] = R_long_direct_emissions.apply(lambda row: 0 if row['Name'].startswith('TRAFO_') and row['Type'] == 'Operation' else row['Value'], axis=1)

In [ ]:
R_long_direct_emissions.to_csv(f'{path_results}impact_scores_direct_emissions.csv', index=False) # [impact / kW(h) or pkm(/h) or tkm(/h)]

### Territorial carbon emissions

In [ ]:
if esm.esm_db is None:
    esm.esm_db = Database(esm.esm_db_name)

In [ ]:
_, contrib_analysis_all_processes, _ = esm.compute_impact_scores(
    methods=methods,
    specific_lcia_abbrev=['m_CCS_all'],
    impact_abbrev=impact_abbrev,
    contribution_analysis='processes',
    contribution_analysis_limit_type='number',
    contribution_analysis_limit=2000,
)

In [ ]:
contrib_analysis_all_processes.to_csv(f'{path_results}contribution_analysis_all_processes_ccst.csv', index=False)

### Add a remaining AoP category (total AoP - climate change) and biogenic CC

In [ ]:
# To skip the impact assessment step
R_long = pd.read_csv(f'{path_results}impact_scores.csv')
R_long_direct_emissions = pd.read_csv(f'{path_results}impact_scores_direct_emissions.csv')
contrib_analysis_all_processes = pd.read_csv(f'{path_results}contribution_analysis_all_processes_ccst.csv')
impact_abbrev = pd.read_csv(path_inputs+'impact_abbrev.csv')

In [ ]:
R_long, _ = add_biogenic_climate_change_to_impact_scores_df(R_long, impact_abbrev)

In [ ]:
R_long_direct_emissions, _ = add_biogenic_climate_change_to_impact_scores_df(R_long_direct_emissions, impact_abbrev)

In [ ]:
R_long, impact_abbrev = add_rhhd_and_reqd_to_impact_scores_df(R_long, impact_abbrev)

In [ ]:
R_long_direct_emissions, _ = add_rhhd_and_reqd_to_impact_scores_df(R_long_direct_emissions, impact_abbrev)

## Create the .mod and .dat files

In [ ]:
metadata = {
    'ecoinvent_version': ei_version,
    'year': year,
    'iam': 'image',
    'ssp_rcp': f'{ssp_rcp}',
}

In [ ]:
specific_lcia_abbrev = ['RHHD', 'REQD', 'm_CCS_all']

In [ ]:
# Snapshot model taken as a 1-step transition model
to_remove = ['TRAIN_FREIGHT_H2_HYBRID_ELD', 'TRAIN_FREIGHT_H2_HYBRID_LD', 'ELEC_EXPORT']
esm.pathway = True
R_long['Year'] = 2025 if year==2023 else 2020
R_long_direct_emissions['Year'] = 2025 if year==2023 else 2020
contrib_analysis_all_processes['Year'] = 2025 if year==2023 else 2020
R_long = R_long[~R_long['Name'].isin(to_remove)]
R_long_direct_emissions = R_long_direct_emissions[~R_long_direct_emissions['Name'].isin(to_remove)]
contrib_analysis_all_processes = contrib_analysis_all_processes[~contrib_analysis_all_processes['act_name'].isin(to_remove)]

In [ ]:
# Create .dat file
esm.normalize_lca_metrics(
    R=R_long,
    mip_gap=1e-6,
    lcia_methods=methods,
    specific_lcia_abbrev=specific_lcia_abbrev,
    impact_abbrev=impact_abbrev,
    path=path_data_lca,
    metadata=metadata,
    file_name='QC_techs_lca',
)

In [ ]:
# Create .dat file for territorial emissions
esm.normalize_lca_metrics(
    assessment_type='territorial emissions',
    R=R_long,
    contrib_processes=contrib_analysis_all_processes,
    mip_gap=1e-6,
    lcia_methods=methods,
    specific_lcia_abbrev=['m_CCS_all'],
    impact_abbrev=impact_abbrev,
    path=path_data_lca,
    metadata=metadata,
    file_name='QC_techs_lca_territorial',
)

In [ ]:
# Create .dat file for direct emissions only
esm.normalize_lca_metrics(
    assessment_type='direct emissions',
    R=R_long,
    R_direct=R_long_direct_emissions,
    mip_gap=1e-6,
    lcia_methods=methods,
    specific_lcia_abbrev=specific_lcia_abbrev,
    impact_abbrev=impact_abbrev,
    path=path_data_lca,
    metadata=metadata,
    file_name='QC_techs_lca_direct',
)

In [ ]:
# Create the .mod file
esm.generate_mod_file_ampl(
    lcia_methods=methods,
    impact_abbrev=impact_abbrev,
    specific_lcia_abbrev=specific_lcia_abbrev,
    path=path_model,
    metadata=metadata,
    file_name='QC_objectives_lca',
)

In [ ]:
# Create the .mod file for direct emissions only
esm.generate_mod_file_ampl(
    assessment_type='direct emissions',
    lcia_methods=methods,
    impact_abbrev=impact_abbrev,
    specific_lcia_abbrev=specific_lcia_abbrev,
    path=path_model,
    metadata=metadata,
    file_name='QC_objectives_lca_direct',
)

In [ ]:
# Create the .mod file for territorial and abroad emissions
esm.generate_mod_file_ampl(
    assessment_type='territorial emissions',
    lcia_methods=methods,
    impact_abbrev=impact_abbrev,
    specific_lcia_abbrev=['m_CCS_all'],
    path=path_model,
    metadata=metadata,
    file_name='QC_objectives_lca_territorial',
)